Train next-day regression models

Build seven inputs, train on earlier years, and save predictions on later years.

In [6]:
%pip install scikit-learn

  Using cached cloudpickle-3.1.2-py3-none-any.whl.metadata (7.1 kB)
   ---------------------------------------- 0.0/8.3 MB ? eta -:--:--
   ----------- ---------------------------- 2.4/8.3 MB 13.4 MB/s eta 0:00:01
   ------------------------ --------------- 5.0/8.3 MB 13.1 MB/s eta 0:00:01
   ------------------------------------ --- 7.6/8.3 MB 13.0 MB/s eta 0:00:01
   ---------------------------------------- 8.3/8.3 MB 12.5 MB/s eta 0:00:00
   ---------------------------------------- 0.0/36.7 MB ? eta -:--:--
   ---- ----------------------------------- 3.9/36.7 MB 21.3 MB/s eta 0:00:02
   --------- ------------------------------ 8.7/36.7 MB 21.5 MB/s eta 0:00:02
   -------------- ------------------------- 13.1/36.7 MB 21.1 MB/s eta 0:00:02
   ------------------- -------------------- 17.6/36.7 MB 20.9 MB/s eta 0:00:01
   ------------------------ --------------- 22.3/36.7 MB 21.4 MB/s eta 0:00:01
   ----------------------------- ---------- 26.7/36.7 MB 21.2 MB/s eta 0:00:01
   ----------

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
arch 7.2.0 requires statsmodels>=0.12, which is not installed.

[notice] A new release of pip is available: 24.3.1 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [7]:
from pathlib import Path
import pandas as pd
from sklearn.linear_model import LinearRegression

# VS Code may start in the project root or in the notebook folder.
ROOT = Path.cwd()
if not (ROOT / "data" / "raw").is_dir():
    ROOT = ROOT.parent
if not (ROOT / "data" / "raw").is_dir():
    raise FileNotFoundError("Open the project containing data/raw, and download the subset first.")
print("Project folder:", ROOT)
OUT = ROOT / "results" / "staged"
OUT.mkdir(parents=True, exist_ok=True)
data = pd.read_csv(ROOT / "data/processed/caravan_selected.csv", parse_dates=["date"])

features = ["streamflow", "flow_yesterday", "flow_two_days_ago", "rainfall",
            "rainfall_3days", "rainfall_7days", "temperature"]
predictions = []
sample_counts = []

Project folder: c:\Users\Pranav Yeturu\Desktop\lab2\Early-Flash-Flood-Prediction


Prepare features and train each basin
Inputs use day t and earlier values; the target is day t+1. Daily reindexing prevents gaps becoming false one-day steps. Training origins end 2009-12-30; test origins start 2010-01-01. Each basin has a separate model. No future observed weather enters the inputs.

In [8]:
for basin, df in data.groupby("basin"):
    # Make features separately for each basin; never mix different rivers.
    df = df.set_index("date").sort_index().asfreq("D")
    df.loc[df["streamflow"] < 0, "streamflow"] = float("nan")

    # At the end of day t, only day-t and earlier inputs may be used.
    df["flow_yesterday"] = df["streamflow"].shift(1)
    df["flow_two_days_ago"] = df["streamflow"].shift(2)
    df["rainfall_3days"] = df["rainfall"].rolling(3).sum()
    df["rainfall_7days"] = df["rainfall"].rolling(7).sum()
    df["target"] = df["streamflow"].shift(-1)

    # Drop missing required observations; never replace discharge gaps with zero.
    clean = df.dropna(subset=features + ["target"])

    # Dec 31, 2009 is excluded: its next-day target belongs to 2010.
    train = clean.loc["1990-01-01":"2009-12-30"]
    test = clean.loc["2010-01-01":"2014-12-30"]

    X_train = train[features]
    y_train = train["target"]
    X_test = test[features]

    model = LinearRegression()
    model.fit(X_train, y_train)
    predicted_flow = model.predict(X_test).clip(min=0)

    # Save both model and baseline predictions on the exact same test rows.
    output = pd.DataFrame({
        "basin": basin,
        "forecast_date": test.index,
        "target_date": test.index + pd.Timedelta(days=1),
        "actual": test["target"].to_numpy(),
        "persistence": test["streamflow"].to_numpy(),
        "linear_regression": predicted_flow,
    })
    predictions.append(output)
    sample_counts.append({"basin": basin, "features": len(features),
                          "training_samples": len(train), "test_samples": len(test)})
    print(basin, "— training:", len(train), "test:", len(test))

camels_01411300 — training: 7304 test: 1571
camels_05495500 — training: 7304 test: 1794
camels_14182500 — training: 7304 test: 1752


Save predictions for evaluation

In [12]:
pd.concat(predictions, ignore_index=True).to_csv(OUT / "test_predictions.csv", index=False)
pd.DataFrame(sample_counts).to_csv(OUT / "sample_counts.csv", index=False)
display(pd.DataFrame(sample_counts))
display(pd.concat(predictions, ignore_index=True).head())

,basin,features,training_samples,test_samples
0,camels_01411300,7,7304,1571
1,camels_05495500,7,7304,1794
2,camels_14182500,7,7304,1752
3,camels_01411300,7,7304,1571
4,camels_05495500,7,7304,1794
5,camels_14182500,7,7304,1752


,basin,forecast_date,target_date,actual,persistence,linear_regression
0,camels_01411300,2010-01-01,2010-01-02,3.44,3.65,3.397049
1,camels_01411300,2010-01-02,2010-01-03,3.07,3.44,3.103453
2,camels_01411300,2010-01-03,2010-01-04,2.85,3.07,2.796306
3,camels_01411300,2010-01-04,2010-01-05,2.70,2.85,2.658627
4,camels_01411300,2010-01-05,2010-01-06,2.52,2.70,2.502218
